# CosyVoice 3-0.5B-RL 声音克隆（Google Colab）

1. 菜单 **代码执行程序 → 更改运行时类型**，硬件加速选 **T4 GPU**（或更好）。
2. 依次运行下面的单元格。需要更新仓库时，单独再跑 **「同步最新代码」** 格（不必重装依赖）。首次会下载约 7GB 权重，大约 10–20 分钟。
3. 运行「启动 WebUI」：在 **Colab 内核里前台**调用，日志实时出现在该格。出现 `http://127.0.0.1:7860` 后可看弹窗或跑 **Colab 端口转发**。中断本格会停掉 WebUI；也可用最下面的「停止 / 退出 WebUI」。

若安装依赖时报 `Failed to build grpcio`：先 **重新运行「克隆/安装」那一格**（会 `git pull` 到已跳过 grpcio/deepspeed 的脚本）。仍异常时：**代码执行程序 → 断开连接并删除运行时**，再从头跑。

免费 Colab 磁盘和会话会过期，断开后需重新下载模型。Colab 当前是 Python 3.13，不能按官方钉死的旧 wheel 安装。

In [ ]:
!nvidia-smi -L || echo "未检测到 GPU，请先把运行时改成 T4 GPU"

**同步最新代码**（随时可再跑）。会 `git fetch` + `git pull --ff-only` 到 `main`。拉完再跑「启动 WebUI」（会自动停掉旧进程再启）。

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/XGVocieClone/.git"):
    !git clone https://github.com/Mr-Yoje/XGVocieClone.git
os.chdir("/content/XGVocieClone")
!git fetch origin
!git checkout main
!git pull --ff-only origin main
!git log -1 --oneline
!chmod +x colab_webui.sh setup_colab.sh 2>/dev/null || true
print("代码已同步到上面的 commit。")

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/XGVocieClone/.git"):
    !git clone https://github.com/Mr-Yoje/XGVocieClone.git
os.chdir("/content/XGVocieClone")
!git pull --ff-only || true
!bash setup_colab.sh

启动 Web 界面。本格直接在 Colab 内核里调用 `webui_clone.main()`，日志会实时出现在下面。应看到 `http://127.0.0.1:7860`。`*.gradio.live` 经常出不来，可再跑 **Colab 端口转发**（需先中断本格的话，页面会一起停；优先看本格是否已弹出窗口）。中断本格会停止 WebUI。

In [ ]:
import os
import subprocess
import sys

os.chdir("/content/XGVocieClone")
subprocess.run(["bash", "colab_webui.sh", "stop"], check=False)
sys.path.insert(0, "/content/XGVocieClone")
sys.argv = [
    "webui_clone.py",
    "--share",
    "--fp16",
    "--force-gpu",
    "--server-name",
    "0.0.0.0",
    "--port",
    "7860",
]
print("在本格前台启动（同一内核，日志会立刻刷出来）", flush=True)
from webui_clone import main

main()

**Colab 端口转发**（`*.gradio.live` 出不来时用）。先保证上面启动格已经在跑，再运行本格，会打开 7860 端口对应的页面。

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(7860)
print("已请求把本机 7860 转到 Colab 窗口。若空白，等启动格出现「监听地址」后再跑一次。")

**停止 / 退出 WebUI**：中断上面的启动格，或运行本格（SIGINT，仍不退出再 SIGTERM / SIGKILL）。停完后若要再用，重新运行「启动」格。

In [ ]:
%cd /content/XGVocieClone
!bash colab_webui.sh stop